In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 0) Quantum core (2-qubit singlet + Born-rule measurement)
# ============================================================

sx = np.array([[0, 1], [1, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
I2 = np.eye(2, dtype=complex)

def bell_singlet():
    # |psi-> = (|01> - |10>)/sqrt(2) in basis |00>,|01>,|10>,|11>
    psi = np.zeros(4, dtype=complex)
    psi[1] = 1/np.sqrt(2)
    psi[2] = -1/np.sqrt(2)
    return psi

def A_op(angle):
    """
    'Polarization-like' analyzer: double-angle mapping on Bloch equator.
    Eigenvalues ±1. Eigenvectors define the two output channels.
    """
    return np.cos(2*angle) * sz + np.sin(2*angle) * sx

def eig_projectors(O):
    w, V = np.linalg.eigh(O)  # columns of V are eigenvectors
    # Identify +1 and -1 eigenvectors robustly
    idx_plus  = np.argmax(w)
    idx_minus = 1 - idx_plus
    v_plus = V[:, idx_plus]
    v_minus = V[:, idx_minus]
    P_plus = np.outer(v_plus, v_plus.conj())
    P_minus = np.outer(v_minus, v_minus.conj())
    return (+1, v_plus, P_plus), (-1, v_minus, P_minus)

def joint_outcome_probs(psi, a, b):
    (sA_p, vA_p, PA_p), (sA_m, vA_m, PA_m) = eig_projectors(A_op(a))
    (sB_p, vB_p, PB_p), (sB_m, vB_m, PB_m) = eig_projectors(A_op(b))

    outs = []
    probs = []
    for sA, vA, PA in [(sA_p, vA_p, PA_p), (sA_m, vA_m, PA_m)]:
        for sB, vB, PB in [(sB_p, vB_p, PB_p), (sB_m, vB_m, PB_m)]:
            P = np.kron(PA, PB)
            p = np.vdot(psi, P @ psi).real
            outs.append((sA, sB, vA, vB, P))
            probs.append(p)

    probs = np.array(probs, dtype=float)
    probs = probs / probs.sum()
    return outs, probs

def sample_joint_outcome(psi, a, b, rng=np.random.default_rng()):
    outs, probs = joint_outcome_probs(psi, a, b)
    k = rng.choice(len(outs), p=probs)
    sA, sB, vA, vB, P = outs[k]
    psi_post = P @ psi
    n = np.linalg.norm(psi_post)
    if n > 0:
        psi_post = psi_post / n
    return dict(
        sA=sA, sB=sB,
        vA=vA, vB=vB,               # local eigenkets for the realized outcomes
        psi_pre=psi.copy(),
        psi_post=psi_post,
        joint_outcomes=outs,
        joint_probs=probs
    )

def reduced_density_matrix(psi, which="A"):
    """
    Reduce a pure 2-qubit state |psi> to rho_A or rho_B.
    Basis ordering is |00>,|01>,|10>,|11> with A=first qubit.
    """
    psi = psi.reshape(2, 2)  # rows = A index, cols = B index
    if which.upper() == "A":
        # rho_A = Tr_B |psi><psi|
        rho = psi @ psi.conj().T
    else:
        # rho_B = Tr_A |psi><psi|
        rho = psi.conj().T @ psi
    return rho

def bloch_vector(rho):
    # r = (Tr[rho sx], Tr[rho sy], Tr[rho sz]) but we only need x,z here;
    sy = np.array([[0, -1j],[1j, 0]], dtype=complex)
    rx = np.trace(rho @ sx).real
    ry = np.trace(rho @ sy).real
    rz = np.trace(rho @ sz).real
    return np.array([rx, ry, rz], dtype=float)

# ============================================================
# 1) "3-phase shell" renderer: map qubit ket [q+, q-] -> αβ(t) -> abc(t)
# ============================================================

def clarke_inv(v_alpha, v_beta):
    """
    Inverse Clarke (no zero-sequence):
      v_a = v_alpha
      v_b = -1/2 v_alpha + sqrt(3)/2 v_beta
      v_c = -1/2 v_alpha - sqrt(3)/2 v_beta
    """
    va = v_alpha
    vb = -0.5*v_alpha + (np.sqrt(3)/2)*v_beta
    vc = -0.5*v_alpha - (np.sqrt(3)/2)*v_beta
    return va, vb, vc

def qubit_to_alphabeta(q, t, omega=1.0):
    """
    Educational embedding:
      v(t) = q_plus * e^{+j ω t} + q_minus * e^{-j ω t}
      where q = [q_plus, q_minus] is a normalized qubit ket.
    Returns real v_alpha(t), v_beta(t).
    """
    q_plus, q_minus = q[0], q[1]
    v = q_plus*np.exp(1j*omega*t) + q_minus*np.exp(-1j*omega*t)
    return v.real, v.imag

def qubit_to_3phase(q, t, omega=1.0):
    v_alpha, v_beta = qubit_to_alphabeta(q, t, omega=omega)
    return clarke_inv(v_alpha, v_beta), (v_alpha, v_beta)

def sample_random_pure_qubit(rng=np.random.default_rng()):
    # Haar-ish sampling of random pure qubit:
    u = rng.uniform(0, 1)
    v = rng.uniform(0, 2*np.pi)
    w = rng.uniform(0, 2*np.pi)
    # |ψ> = cos(θ/2)|0> + e^{iφ} sin(θ/2)|1>
    theta = np.arccos(1 - 2*u)   # theta ~ sin(theta) distribution
    phi = v
    ket = np.array([np.cos(theta/2), np.exp(1j*phi)*np.sin(theta/2)], dtype=complex)
    # global phase not important
    return ket / np.linalg.norm(ket)

# ============================================================
# 2) Visualization helpers
# ============================================================

def plot_joint_prob_table(ax, outs, probs, title="Joint outcome probabilities"):
    # Arrange probs into a 2x2 grid for (sA, sB) = (+,+), (+,-), (-,+), (-,-)
    # We'll map indices explicitly:
    p_map = {}
    for (sA, sB, *_), p in zip(outs, probs):
        p_map[(sA, sB)] = p

    grid = np.array([
        [p_map[(+1,+1)], p_map[(+1,-1)]],
        [p_map[(-1,+1)], p_map[(-1,-1)]]
    ])

    im = ax.imshow(grid, aspect='equal')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels([r"$s_B=+1$", r"$s_B=-1$"])
    ax.set_yticklabels([r"$s_A=+1$", r"$s_A=-1$"])
    ax.set_title(title)

    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{grid[i,j]:.3f}", ha="center", va="center", color="white" if grid[i,j] > 0.5 else "black")

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

def plot_alphabeta(ax, v_alpha, v_beta, title="αβ trajectory"):
    ax.plot(v_alpha, v_beta, lw=1.0)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.3)
    ax.set_xlabel(r"$v_\alpha$")
    ax.set_ylabel(r"$v_\beta$")
    ax.set_title(title)

def plot_three_phase(ax, va, vb, vc, t, title="3-phase (abc)"):
    ax.plot(t, va, lw=1.0, label="Va")
    ax.plot(t, vb, lw=1.0, label="Vb")
    ax.plot(t, vc, lw=1.0, label="Vc")
    ax.grid(True, alpha=0.3)
    ax.set_title(title)
    ax.set_xlabel("t")
    ax.legend(frameon=False, fontsize=8)

def plot_bloch_equator_axes(ax, angles, labels, title="Analyzer axes on x–z equator (double-angle)"):
    # Draw unit circle in x–z plane (we'll plot z vertical, x horizontal)
    th = np.linspace(0, 2*np.pi, 400)
    ax.plot(np.cos(th), np.sin(th), lw=1.0)  # (x,z)

    for ang, lab in zip(angles, labels):
        # measurement axis n(a) = (sin2a, 0, cos2a) => (x,z) = (sin2a, cos2a)
        x = np.sin(2*ang)
        z = np.cos(2*ang)
        ax.arrow(0, 0, x, z, head_width=0.05, head_length=0.07, length_includes_head=True)
        ax.text(1.08*x, 1.08*z, lab, ha="center", va="center", fontsize=9)

    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-1.15, 1.15)
    ax.set_ylim(-1.15, 1.15)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("x")
    ax.set_ylabel("z")
    ax.set_title(title)

def plot_pre_state_ensemble(ax_ab, ax_abc, t, omega, n_samples=30, rng=np.random.default_rng()):
    """
    For the singlet, local reduced state is maximally mixed (I/2), so there's no single ket.
    Show this by plotting an ensemble of random pure kets consistent with "unknown local state".
    """
    for _ in range(n_samples):
        q = sample_random_pure_qubit(rng=rng)
        (va, vb, vc), (v_alpha, v_beta) = qubit_to_3phase(q, t, omega=omega)
        ax_ab.plot(v_alpha, v_beta, lw=0.6, alpha=0.15)
        ax_abc.plot(t, va, lw=0.6, alpha=0.10)
        ax_abc.plot(t, vb, lw=0.6, alpha=0.10)
        ax_abc.plot(t, vc, lw=0.6, alpha=0.10)

    ax_ab.set_title("Before collapse: local state is mixed → ensemble of possible αβ ellipses")
    ax_ab.set_aspect('equal', adjustable='box')
    ax_ab.grid(True, alpha=0.3)
    ax_ab.set_xlabel(r"$v_\alpha$")
    ax_ab.set_ylabel(r"$v_\beta$")

    ax_abc.set_title("Before collapse: ensemble of possible 3-phase waveforms (not a unique waveform)")
    ax_abc.grid(True, alpha=0.3)
    ax_abc.set_xlabel("t")

# ============================================================
# 3) Demo: one CHSH setting pair with BEFORE/AFTER collapse visuals
# ============================================================

rng = np.random.default_rng(7)

# Standard CHSH angles (in radians)
a0, a1 = 0.0, np.pi/4
b0, b1 = np.pi/8, -np.pi/8

# Choose one setting pair to visualize
a = a0
b = b0

psi = bell_singlet()

# Sample ONE joint measurement outcome for this (a,b)
res = sample_joint_outcome(psi, a, b, rng=rng)
sA, sB = res["sA"], res["sB"]
vA_ket, vB_ket = res["vA"], res["vB"]

# Reduced states before and after (pure state -> reduced mixed/pure-ish depending on post)
rhoA_pre  = reduced_density_matrix(res["psi_pre"],  "A")
rhoB_pre  = reduced_density_matrix(res["psi_pre"],  "B")
rhoA_post = reduced_density_matrix(res["psi_post"], "A")
rhoB_post = reduced_density_matrix(res["psi_post"], "B")

# Time base for rendering "3-phase shell"
N = 1200
t = np.linspace(0, 2*np.pi, N, endpoint=False)
omega = 1.0

# After collapse, for the realized outcomes, the local states should match the measurement eigenkets
# We'll render those eigenkets directly (cleanest educationally).
(alice_abc, alice_ab) = qubit_to_3phase(vA_ket, t, omega=omega)
(bob_abc,   bob_ab)   = qubit_to_3phase(vB_ket, t, omega=omega)

# ============================================================
# Figure 1: analyzers + joint probabilities + Bloch vectors before/after
# ============================================================

fig1, axs = plt.subplots(1, 3, figsize=(13, 4))

plot_bloch_equator_axes(
    axs[0],
    angles=[a, b],
    labels=[f"A (a={np.degrees(a):.1f}°)", f"B (b={np.degrees(b):.1f}°)"],
    title="Analyzer axes (double-angle mapping)"
)

plot_joint_prob_table(
    axs[1],
    res["joint_outcomes"],
    res["joint_probs"],
    title=f"Joint probs for one setting pair\n(realized outcome: sA={sA:+d}, sB={sB:+d})"
)

# Bloch vectors (pre should be ~0 for singlet reductions)
rA_pre, rB_pre = bloch_vector(rhoA_pre), bloch_vector(rhoB_pre)
rA_post, rB_post = bloch_vector(rhoA_post), bloch_vector(rhoB_post)

axs[2].axhline(0, lw=1.0)
axs[2].plot([0,1,2], [rA_pre[0], rA_post[0], np.nan], marker='o', label="Alice rx")
axs[2].plot([0,1,2], [rA_pre[2], rA_post[2], np.nan], marker='o', label="Alice rz")
axs[2].plot([0,1,2], [rB_pre[0], rB_post[0], np.nan], marker='o', label="Bob rx")
axs[2].plot([0,1,2], [rB_pre[2], rB_post[2], np.nan], marker='o', label="Bob rz")
axs[2].set_xticks([0,1])
axs[2].set_xticklabels(["pre", "post"])
axs[2].set_title("Local Bloch components (x,z)\n(pre is mixed, post is conditioned)")
axs[2].grid(True, alpha=0.3)
axs[2].legend(frameon=False, fontsize=8)

plt.tight_layout()
plt.show()

# ============================================================
# Figure 2: BEFORE collapse (ensemble) vs AFTER collapse (one realized state)
# ============================================================

fig2, axs = plt.subplots(2, 2, figsize=(13, 7))

# BEFORE: ensemble (because local reduced state is mixed)
plot_pre_state_ensemble(axs[0,0], axs[0,1], t, omega, n_samples=40, rng=rng)

# AFTER: Alice and Bob realized eigenkets rendered as αβ and abc
vA_alpha, vA_beta = alice_ab
vB_alpha, vB_beta = bob_ab

(vaA, vbA, vcA) = alice_abc
(vaB, vbB, vcB) = bob_abc

plot_alphabeta(axs[1,0], vA_alpha, vA_beta,
               title=f"After collapse: Alice αβ trajectory (sA={sA:+d})")
plot_alphabeta(axs[1,1], vB_alpha, vB_beta,
               title=f"After collapse: Bob αβ trajectory (sB={sB:+d})")

plt.tight_layout()
plt.show()

fig3, axs = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
plot_three_phase(axs[0], vaA, vbA, vcA, t, title=f"Alice 3-phase after collapse (sA={sA:+d})")
plot_three_phase(axs[1], vaB, vbB, vcB, t, title=f"Bob   3-phase after collapse (sB={sB:+d})")
plt.tight_layout()
plt.show()

# ============================================================
# 4) Full CHSH with strict ±1 outcomes + convergence plot
# ============================================================

def estimate_E_and_S(n_trials=500, rng=np.random.default_rng(0)):
    psi = bell_singlet()
    settings = [(a0,b0), (a0,b1), (a1,b0), (a1,b1)]
    Es = []

    for (aa, bb) in settings:
        vals = []
        for _ in range(n_trials):
            r = sample_joint_outcome(psi, aa, bb, rng=rng)
            vals.append(r["sA"] * r["sB"])
        Es.append(np.mean(vals))

    E00, E01, E10, E11 = Es
    S = abs(E00 + E01 + E10 - E11)
    return S, (E00, E01, E10, E11)

# Convergence of S vs trials
trial_grid = np.unique(np.logspace(2, 5, 18).astype(int))  # 1e2..1e5
S_vals = []
for n in trial_grid:
    S, Es = estimate_E_and_S(n_trials=int(n), rng=rng)
    S_vals.append(S)

fig4, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(trial_grid, S_vals, marker="o", lw=1.0)
ax.axhline(2.0, ls="--", lw=1.0)
ax.axhline(2*np.sqrt(2), ls="-", lw=1.0)
ax.set_xscale("log")
ax.grid(True, alpha=0.3)
ax.set_xlabel("Number of trials per setting (log scale)")
ax.set_ylabel("Estimated CHSH S")
ax.set_title("Strict ±1 CHSH with Born-rule sampling (converges to 2√2)")
plt.tight_layout()
plt.show()
